# 🤖 Buổi 6/9 — Model Training (Linear Models)
### Huấn luyện và so sánh các mô hình hồi quy tuyến tính!

---

> **📍 Bạn đang ở đây trong lộ trình 9 buổi:**
> ```
> Buổi 1 ✅ Giới thiệu dự án & lộ trình
> Buổi 2 ✅ Setup môi trường & khám phá SalePrice
> Buổi 3 ✅ EDA — Phân tích missing, tương quan, outliers
> Buổi 4 ✅ Data Cleaning & Preprocessing
> Buổi 5 ✅ Feature Engineering
> Buổi 6 🔄 Model Training (Linear Models)            ← BẠN ĐANG Ở ĐÂY
> Buổi 7 ⏳ Tree-Based Models & Đánh Giá (RF, XGBoost, LightGBM)
> Buổi 8 ⏳ Submit Kaggle (tạo file & nộp bài)
> Buổi 9 ⏳ Quiz Tốt Nghiệp (20 câu ôn tập)
> ```

---

### 📋 Nội Dung Buổi 6

| # | Task | Nội dung | Kết quả |
|---|---|---|---|
| 0 | ♻️ Setup | Rebuild pipeline từ Buổi 5 | X_train, X_test, y sẵn sàng |
| 1 | 📐 Linear Regression | Baseline model, hiểu overfitting | RMSE baseline |
| 2 | 🟦 Ridge (L2) | Regularization bình phương, RidgeCV | RMSE cải thiện |
| 3 | 🟧 Lasso (L1) | Feature selection tự động, LassoCV | RMSE + sparse coefficients |
| 4 | 🔀 ElasticNet | Kết hợp L1+L2 | So sánh 4 models |
| 5 | 📊 So sánh Models | Bar chart RMSE, Actual vs Predicted | Chọn model tốt nhất |
| 6 | 🔍 Phân tích Hệ số | Top features theo Ridge & Lasso | Hiểu model |

---

### 🔗 Kết Nối Với Buổi 5

Ở Buổi 5, chúng ta đã tạo ra `X_train` (~1458 × 215) và `X_test` (~1459 × 215) đã scale chuẩn hoá.  
Buổi 6 này chúng ta sẽ **đưa dữ liệu đó vào các mô hình** để dự đoán giá nhà!

```
Pipeline hiện tại:
load → fill → rm outlier → feature engineering → ordinal enc
→ one-hot → variance filter → StandardScaler
                                        ↓
                              X_train, X_test, y  ← ĐẦU VÀO CHO MODEL
```

> 🎯 **Mục tiêu Buổi 6:** RMSE(log) ≤ 0.12 với cross-validation 5-fold

In [ ]:
# ============================================================
# ♻️ SETUP — Import thư viện & rebuild pipeline từ Buổi 5
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, RidgeCV, LassoCV
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import r2_score, mean_squared_error

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:.3f}')

# ── Load dữ liệu gốc ──────────────────────────────────────
train_df = pd.read_csv('data/train.csv')
test_df  = pd.read_csv('data/test.csv')

train_id = train_df['Id'].copy()
test_id  = test_df['Id'].copy()
y = np.log1p(train_df['SalePrice'])

train_df.drop(['Id', 'SalePrice'], axis=1, inplace=True)
test_df.drop(['Id'], axis=1, inplace=True)

ntrain = len(train_df)
ntest  = len(test_df)
all_data = pd.concat([train_df, test_df], axis=0, ignore_index=True)

# ── Fill missing ──────────────────────────────────────────
none_cols = ['PoolQC','MiscFeature','Alley','Fence','FireplaceQu',
             'GarageType','GarageFinish','GarageQual','GarageCond',
             'BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2','MasVnrType']
zero_cols = ['GarageYrBlt','GarageArea','GarageCars','MasVnrArea',
             'BsmtFinSF1','BsmtFinSF2','BsmtUnfSF','TotalBsmtSF','BsmtFullBath','BsmtHalfBath']
mode_cols = ['MSZoning','Utilities','Functional','Electrical',
             'KitchenQual','Exterior1st','Exterior2nd','SaleType']

for col in none_cols: all_data[col] = all_data[col].fillna('None')
for col in zero_cols: all_data[col] = all_data[col].fillna(0)
all_data['LotFrontage'] = (all_data.groupby('Neighborhood')['LotFrontage']
                           .transform(lambda x: x.fillna(x.median())))
for col in mode_cols: all_data[col] = all_data[col].fillna(all_data[col].mode()[0])

# ── Xoá outlier ───────────────────────────────────────────
outlier_mask = (all_data['GrLivArea'][:ntrain] > 4000) & (np.expm1(y) < 300_000)
outlier_idx  = outlier_mask[outlier_mask].index.tolist()
all_data.drop(index=outlier_idx, inplace=True)
all_data.reset_index(drop=True, inplace=True)
y.drop(index=outlier_idx, inplace=True)
y.reset_index(drop=True, inplace=True)
ntrain = ntrain - len(outlier_idx)

# ── Feature Engineering ──────────────────────────────────
all_data['TotalSF']      = (all_data['TotalBsmtSF'] + all_data['1stFlrSF'] + all_data['2ndFlrSF'])
all_data['TotalPorchSF'] = (all_data['OpenPorchSF'] + all_data['3SsnPorch'] +
                             all_data['EnclosedPorch'] + all_data['ScreenPorch'] + all_data['WoodDeckSF'])
all_data['TotalBath']    = (all_data['FullBath'] + 0.5*all_data['HalfBath'] +
                             all_data['BsmtFullBath'] + 0.5*all_data['BsmtHalfBath'])
all_data['OverallScore'] = all_data['OverallQual'] * all_data['OverallCond']
all_data['HasGarage']    = (all_data['GarageArea'] > 0).astype(int)
all_data['HasBsmt']      = (all_data['TotalBsmtSF'] > 0).astype(int)

all_data['HouseAge']  = all_data['YrSold'] - all_data['YearBuilt']
all_data['RemodAge']  = all_data['YrSold'] - all_data['YearRemodAdd']
all_data['GarageAge'] = (all_data['YrSold'] - all_data['GarageYrBlt']).clip(lower=0)
all_data['IsNew']     = (all_data['YearBuilt'] >= 2000).astype(int)
all_data['HasRemod']  = (all_data['YearRemodAdd'] != all_data['YearBuilt']).astype(int)
all_data['SoldAsNew'] = (all_data['YrSold'] == all_data['YearBuilt']).astype(int)

top_fe = ['OverallQual', 'GrLivArea', 'TotalSF']
for feat in top_fe:
    all_data[f'{feat}_sq']   = all_data[feat] ** 2
    all_data[f'{feat}_sqrt'] = np.sqrt(all_data[feat])

for feat in ['GrLivArea', 'TotalSF', 'LotArea', '1stFlrSF', 'TotalPorchSF']:
    if feat in all_data.columns:
        all_data[f'{feat}_log'] = np.log1p(all_data[feat])

# ── Ordinal Encoding ──────────────────────────────────────
ordinal_mappings = {
    'ExterQual':    ['None','Po','Fa','TA','Gd','Ex'],
    'ExterCond':    ['None','Po','Fa','TA','Gd','Ex'],
    'BsmtQual':     ['None','Po','Fa','TA','Gd','Ex'],
    'BsmtCond':     ['None','Po','Fa','TA','Gd','Ex'],
    'BsmtExposure': ['None','No','Mn','Av','Gd'],
    'BsmtFinType1': ['None','Unf','LwQ','Rec','BLQ','ALQ','GLQ'],
    'BsmtFinType2': ['None','Unf','LwQ','Rec','BLQ','ALQ','GLQ'],
    'HeatingQC':    ['None','Po','Fa','TA','Gd','Ex'],
    'KitchenQual':  ['None','Po','Fa','TA','Gd','Ex'],
    'FireplaceQu':  ['None','Po','Fa','TA','Gd','Ex'],
    'GarageFinish': ['None','Unf','RFn','Fin'],
    'GarageQual':   ['None','Po','Fa','TA','Gd','Ex'],
    'GarageCond':   ['None','Po','Fa','TA','Gd','Ex'],
    'PoolQC':       ['None','Fa','TA','Gd','Ex'],
    'Fence':        ['None','MnWw','GdWo','MnPrv','GdPrv'],
    'Functional':   ['Sal','Sev','Maj2','Maj1','Mod','Min2','Min1','Typ'],
    'LandSlope':    ['Sev','Mod','Gtl'],
    'LotShape':     ['IR3','IR2','IR1','Reg'],
    'PavedDrive':   ['N','P','Y'],
}
for col, order in ordinal_mappings.items():
    mapping_dict = {val: i for i, val in enumerate(order)}
    all_data[col] = all_data[col].map(mapping_dict).fillna(0).astype(int)

# ── One-Hot + Variance Filter + Scale ────────────────────
cat_cols     = all_data.select_dtypes(include=['object']).columns.tolist()
all_data_enc = pd.get_dummies(all_data, columns=cat_cols, drop_first=True)

X_train_raw = all_data_enc[:ntrain].copy()
X_test_raw  = all_data_enc[ntrain:].copy()
X_test_raw.reset_index(drop=True, inplace=True)

selector = VarianceThreshold(threshold=0.01)
selector.fit(X_train_raw)
selected_mask = selector.get_support()

X_train_sel = X_train_raw.loc[:, selected_mask].copy()
X_test_sel  = X_test_raw.loc[:, selected_mask].copy()

scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train_sel),
                        columns=X_train_sel.columns, index=X_train_sel.index)
X_test  = pd.DataFrame(scaler.transform(X_test_sel),
                        columns=X_test_sel.columns, index=X_test_sel.index)

print("✅ Pipeline Buổi 5 đã rebuild thành công!")
print(f"   X_train : {X_train.shape}")
print(f"   X_test  : {X_test.shape}")
print(f"   y       : {y.shape}   (log1p SalePrice)")
print(f"   ntrain  : {ntrain}")
print()
print("⏭️ Sẵn sàng train models!")

**Expected Output:**

![img6-setup](images/img_buoi6/img6-1.png)

---

## 📐 Task 1 — Cross-Validation & Baseline (Linear Regression)

---

### 🤔 Tại Sao Cần Cross-Validation?

Khi train model, chúng ta cần **đánh giá** xem model tốt đến đâu. Nếu chỉ đo trên chính data đã train → **overfit**, điểm số không phản ánh thực tế.

```
Vấn đề với train/test split đơn giản:
   Nếu hên chọn được test tốt → điểm cao giả tạo
   Nếu xui → điểm thấp dù model ổn

Cross-validation 5-fold giải quyết:
   Chia train thành 5 phần bằng nhau
   Lần lượt dùng mỗi phần làm "validation" → 5 điểm số
   Lấy trung bình → ổn định, đáng tin hơn
```

```
Fold 1: [VAL] [---] [---] [---] [---]
Fold 2: [---] [VAL] [---] [---] [---]
Fold 3: [---] [---] [VAL] [---] [---]
Fold 4: [---] [---] [---] [VAL] [---]
Fold 5: [---] [---] [---] [---] [VAL]
         ↑ mỗi lần train trên 4 fold, validate trên 1 fold
```

### 📏 Metric: RMSE(log)

$$\text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(\hat{y}_i - y_i)^2}$$

Vì `y = log1p(SalePrice)`, RMSE đo **sai số trong không gian log** — tương đương sai số **phần trăm** giá nhà.

> **Mục tiêu:** RMSE(log) < 0.12 ≈ sai số ~12% giá thực tế

In [ ]:
# ============================================================
# 📐 TASK 1 — Helper Function & Linear Regression Baseline
# ============================================================

def rmse_cv(model, X, y, cv=5):
    """Tính cross-validated RMSE (5-fold theo mặc định)"""
    kf = KFold(n_splits=cv, shuffle=True, random_state=42)
    scores = cross_val_score(model, X, y,
                             scoring='neg_mean_squared_error',
                             cv=kf)
    return np.sqrt(-scores)

# ── Linear Regression — Baseline ──────────────────────────
print("=" * 55)
print("📐 LINEAR REGRESSION — Baseline (không regularization)")
print("=" * 55)

lr = LinearRegression()
lr_scores = rmse_cv(lr, X_train, y)

print(f"   CV RMSE (5-fold): {lr_scores.mean():.5f} ± {lr_scores.std():.5f}")
print(f"   Các fold       : {[round(s,5) for s in lr_scores]}")
print()

# Fit để xem R² và train RMSE
lr.fit(X_train, y)
y_pred_lr    = lr.predict(X_train)
train_rmse_lr = np.sqrt(mean_squared_error(y, y_pred_lr))
r2_lr         = r2_score(y, y_pred_lr)

print(f"   Train RMSE : {train_rmse_lr:.5f}  ← nhỏ hơn CV RMSE nhiều → OVERFIT!")
print(f"   R² (train) : {r2_lr:.5f}")
print()
print("⚠️  Nhận xét: Linear Regression thường bị OVERFIT khi có 200+ features")
print("   Train RMSE thấp nhưng CV RMSE cao → model 'học thuộc lòng' train data")
print("   → Cần REGULARIZATION để giải quyết: Ridge, Lasso, ElasticNet")

---

### 🔬 Giải Thích — Linear Regression & Overfitting

---

```python
def rmse_cv(model, X, y, cv=5):
    scores = cross_val_score(model, X, y, scoring='neg_mean_squared_error', cv=kf)
    return np.sqrt(-scores)
```

| Tham số | Ý nghĩa |
|---|---|
| `scoring='neg_mean_squared_error'` | sklearn trả về giá trị **âm** (để tối đa hoá); ta lấy `-scores` để về dương |
| `np.sqrt(...)` | Chuyển MSE → RMSE (cùng đơn vị với target) |
| `KFold(shuffle=True, random_state=42)` | Shuffle để tránh bias theo thứ tự, seed cố định để tái tạo |

**Tại sao Linear Regression bị overfit với nhiều features?**

```
Hàm mất mát Linear Regression:
   Loss = Σ(y_pred - y_true)²

Với 215 features và 1458 mẫu → model có QUÁ NHIỀU tự do
→ Nó "nhớ" dữ liệu train thay vì "học" pattern chung
→ Hệ số (coefficients) có thể rất lớn, rất không ổn định
```

**Giải pháp: Thêm penalty vào hàm mất mát:**

| Model | Hàm mất mát | Hiệu ứng |
|---|---|---|
| Linear Regression | $\text{MSE}$ | Không có kiểm soát |
| Ridge (L2) | $\text{MSE} + \alpha \sum w_i^2$ | Hệ số nhỏ lại, không về 0 |
| Lasso (L1) | $\text{MSE} + \alpha \sum \|w_i\|$ | Một số hệ số = 0 (feature selection) |
| ElasticNet | $\text{MSE} + \alpha_1 \sum \|w_i\| + \alpha_2 \sum w_i^2$ | Kết hợp cả hai |

---

## 🟦 Task 2 — Ridge Regression (L2 Regularization)

---

### 🤔 Ridge Hoạt Động Như Thế Nào?

```
Hàm mất mát Ridge:
   Loss = MSE + α × Σwᵢ²
              ↑
         L2 Penalty: "phạt" hệ số lớn
```

**Tác dụng của alpha (α):**
- `α = 0` → giống Linear Regression (không penalty)
- `α nhỏ` → penalty nhẹ, hệ số vẫn lớn
- `α lớn` → penalty mạnh, hệ số bị ép về gần 0 (nhưng KHÔNG bằng đúng 0)

**RidgeCV**: sklearn tự động thử nhiều giá trị alpha và chọn tốt nhất qua CV.

> 💡 **Đặc điểm chính của Ridge:** Giữ **tất cả features** (coeff ≠ 0), chỉ làm nhỏ chúng lại → tốt khi nhiều features đều có ích.

In [ ]:
# ============================================================
# 🟦 TASK 2 — RIDGE REGRESSION
# ============================================================

print("=" * 55)
print("🟦 RIDGE REGRESSION (L2 Regularization)")
print("=" * 55)
print()

# ── Thử nhiều giá trị alpha ────────────────────────────────
alphas_to_try = [0.1, 1, 5, 10, 20, 50, 100, 200]
print(f"   {'Alpha':>8} | {'CV RMSE':>10} | {'± Std':>8}")
print("   " + "─" * 35)

ridge_results = {}
for alpha in alphas_to_try:
    scores = rmse_cv(Ridge(alpha=alpha), X_train, y)
    ridge_results[alpha] = scores.mean()
    print(f"   {alpha:>8.1f} | {scores.mean():>10.5f} | ±{scores.std():.5f}")

best_alpha_ridge = min(ridge_results, key=ridge_results.get)
print(f"\n   → Alpha tốt nhất thủ công: {best_alpha_ridge}")
print()

# ── RidgeCV — tự động chọn alpha tối ưu ───────────────────
alphas_ridge = np.logspace(-2, 3, 100)   # 100 giá trị từ 0.01 đến 1000
ridge_cv = RidgeCV(alphas=alphas_ridge, cv=5, scoring='neg_mean_squared_error')
ridge_cv.fit(X_train, y)

print(f"   RidgeCV best alpha : {ridge_cv.alpha_:.4f}")

# Đánh giá với alpha tốt nhất
ridge_best = Ridge(alpha=ridge_cv.alpha_)
ridge_scores = rmse_cv(ridge_best, X_train, y)
print(f"   CV RMSE (5-fold)   : {ridge_scores.mean():.5f} ± {ridge_scores.std():.5f}")
print()

# Fit model cuối
ridge_best.fit(X_train, y)
y_pred_ridge = ridge_best.predict(X_train)
print(f"   Train RMSE : {np.sqrt(mean_squared_error(y, y_pred_ridge)):.5f}")
print(f"   R² (train) : {r2_score(y, y_pred_ridge):.5f}")
print(f"   Số features: {(ridge_best.coef_ != 0).sum()} / {X_train.shape[1]} (Ridge giữ TẤT CẢ features)")

---

### 🔬 Giải Thích — Ridge & Alpha Tuning

---

```python
alphas_ridge = np.logspace(-2, 3, 100)
ridge_cv = RidgeCV(alphas=alphas_ridge, cv=5)
ridge_cv.fit(X_train, y)
print(ridge_cv.alpha_)   # alpha tối ưu
```

| Lệnh | Ý nghĩa |
|---|---|
| `np.logspace(-2, 3, 100)` | Tạo 100 giá trị theo tỉ lệ log: 0.01, 0.02, ..., 100, ..., 1000 |
| `RidgeCV(alphas=..., cv=5)` | Tự động thử tất cả alpha, giữ alpha có CV RMSE tốt nhất |
| `.alpha_` | Truy cập alpha đã được chọn sau khi fit |

**Tại sao dùng logspace thay vì linspace?**

```
linspace(0.1, 100, 100): [0.1, 1.1, 2.1, ..., 100]
→ Quá nhiều giá trị ở vùng lớn, ít ở vùng nhỏ

logspace(-2, 3, 100): [0.01, 0.02, 0.05, ..., 50, 100, ..., 1000]
→ Phân bố đều theo tỉ lệ → tìm alpha tốt hơn trong cả hai vùng
```

**🌐 Khi Nào Dùng Ridge?**

| Tình huống | Lý do dùng Ridge |
|---|---|
| Nhiều features, đều có ích | Giữ tất cả features (coeff ≠ 0) |
| Features tương quan cao (multicollinearity) | Ridge xử lý tốt hơn Linear Regression |
| Muốn model ổn định | L2 làm hệ số nhỏ đều, ít nhạy cảm với noise |

---

## 🟧 Task 3 — Lasso Regression (L1 Regularization)

---

### 🤔 Lasso Khác Ridge Ở Điểm Gì?

```
Hàm mất mát Lasso:
   Loss = MSE + α × Σ|wᵢ|
              ↑
         L1 Penalty: giá trị tuyệt đối → một số coeff = 0 chính xác
```

**Điểm đặc biệt của Lasso:**
- Khi α đủ lớn, nhiều hệ số = **đúng bằng 0** → tự động loại features!
- Hoạt động như **feature selection tích hợp sẵn**
- Chỉ giữ những features thực sự quan trọng

```
Ridge:  w₁=0.001, w₂=0.002, w₃=0.0001  (TẤT CẢ ≠ 0, chỉ nhỏ lại)
Lasso:  w₁=0.05,  w₂=0.08,  w₃=0       (w₃ = 0 chính xác, loại khỏi model)
```

> 💡 **Khi nào Lasso tốt hơn Ridge?** Khi bạn nghi ngờ chỉ một số ít features thực sự quan trọng — Lasso sẽ tự "gạt" phần còn lại ra.

In [ ]:
# ============================================================
# 🟧 TASK 3 — LASSO REGRESSION
# ============================================================

print("=" * 55)
print("🟧 LASSO REGRESSION (L1 Regularization)")
print("=" * 55)
print()

# ── LassoCV — tự động tìm alpha tối ưu ────────────────────
alphas_lasso = np.logspace(-4, 0, 100)   # từ 0.0001 đến 1
lasso_cv = LassoCV(alphas=alphas_lasso, cv=5, max_iter=10000, random_state=42)
lasso_cv.fit(X_train, y)

print(f"   LassoCV best alpha : {lasso_cv.alpha_:.6f}")

# Đánh giá với alpha tốt nhất
lasso_best = Lasso(alpha=lasso_cv.alpha_, max_iter=10000)
lasso_scores = rmse_cv(lasso_best, X_train, y)
print(f"   CV RMSE (5-fold)   : {lasso_scores.mean():.5f} ± {lasso_scores.std():.5f}")
print()

# Fit model cuối
lasso_best.fit(X_train, y)
y_pred_lasso = lasso_best.predict(X_train)

# Đếm features được giữ lại
nonzero_count = (lasso_best.coef_ != 0).sum()
zero_count    = (lasso_best.coef_ == 0).sum()

print(f"   Train RMSE : {np.sqrt(mean_squared_error(y, y_pred_lasso)):.5f}")
print(f"   R² (train) : {r2_score(y, y_pred_lasso):.5f}")
print()
print(f"   🔍 Feature Selection by Lasso:")
print(f"      Giữ lại (coeff ≠ 0) : {nonzero_count:>4} features")
print(f"      Loại bỏ (coeff = 0) : {zero_count:>4} features")
print(f"      Tổng features       : {X_train.shape[1]:>4} features")
print()

# Top features quan trọng nhất theo Lasso
lasso_coef = pd.Series(lasso_best.coef_, index=X_train.columns)
top_lasso  = lasso_coef.abs().sort_values(ascending=False).head(10)
print("   Top 10 features quan trọng nhất (theo Lasso):")
for feat, coef in top_lasso.items():
    sign = '+' if lasso_coef[feat] > 0 else '-'
    bar  = '█' * int(coef * 100)
    print(f"      {feat:<35} {sign}{coef:.4f}  {bar}")

---

### 🔬 Giải Thích — Lasso & Feature Selection

---

**Tại sao L1 tạo ra hệ số = 0 còn L2 thì không?**

```
Hình học giải thích:
   L2 (Ridge): constraint là hình cầu → điểm tối ưu ít khi chạm trục
   L1 (Lasso): constraint là hình thoi → điểm tối ưu thường ở đỉnh thoi → w = 0

Đây là lý do Lasso "thưa" (sparse) còn Ridge thì không
```

```python
lasso_cv = LassoCV(alphas=alphas_lasso, cv=5, max_iter=10000)
lasso_cv.fit(X_train, y)
```

| Tham số | Ý nghĩa |
|---|---|
| `max_iter=10000` | Lasso dùng coordinate descent — cần nhiều iteration hơn Ridge |
| `LassoCV` | Tự động grid search alpha và chọn tốt nhất bằng cross-validation |

**🌐 Ứng dụng Lasso ngoài bất động sản**

```python
# Gene expression (bioinformatics): 20,000 genes, chỉ ~100 quan trọng
# → Lasso tự chọn 100 genes, loại 19,900 còn lại
lasso_gene = Lasso(alpha=0.001).fit(X_gene, y_disease)
important_genes = X_gene.columns[lasso_gene.coef_ != 0]
```

> 💡 **Quy tắc ngón tay cái:** Khi không biết dùng Ridge hay Lasso → thử cả hai! ElasticNet kết hợp cả hai thường cho kết quả ổn định nhất.

---

## 🔀 Task 4 — ElasticNet (L1 + L2)

---

### 🤔 ElasticNet Là Gì?

ElasticNet kết hợp cả Ridge và Lasso:

$$\text{Loss} = \text{MSE} + \alpha \left[ \rho \sum|w_i| + \frac{1-\rho}{2} \sum w_i^2 \right]$$

| Tham số | Ý nghĩa |
|---|---|
| `alpha` | Cường độ regularization tổng thể |
| `l1_ratio` (= ρ) | Tỉ lệ L1: `l1_ratio=1` → thuần Lasso, `l1_ratio=0` → thuần Ridge |

**Khi nào ElasticNet tốt nhất?**
- Khi không chắc chắn nên dùng Ridge hay Lasso
- Khi dataset có nhiều features correlated cao (Ridge tốt) + nhiều noise features (Lasso tốt)

In [ ]:
# ============================================================
# 🔀 TASK 4 — ELASTICNET
# ============================================================

print("=" * 55)
print("🔀 ELASTICNET (L1 + L2 Combined)")
print("=" * 55)
print()

# ── Thử các combination l1_ratio + alpha ──────────────────
configs = [
    (0.0005, 0.5, 'cân bằng L1/L2'),
    (0.0005, 0.7, 'nghiêng Lasso'),
    (0.0005, 0.3, 'nghiêng Ridge'),
    (0.001,  0.5, 'alpha cao hơn'),
]

print(f"   {'Alpha':>8} | {'l1_ratio':>9} | {'Config':<18} | {'CV RMSE':>10}")
print("   " + "─" * 58)

elastic_results = {}
for alpha, l1_ratio, desc in configs:
    elastic = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=10000)
    scores  = rmse_cv(elastic, X_train, y)
    elastic_results[(alpha, l1_ratio)] = scores.mean()
    print(f"   {alpha:>8.4f} | {l1_ratio:>9.1f} | {desc:<18} | {scores.mean():>10.5f}")

# Chọn config tốt nhất
best_config   = min(elastic_results, key=elastic_results.get)
best_alpha_e, best_l1 = best_config
print(f"\n   → Config tốt nhất: alpha={best_alpha_e}, l1_ratio={best_l1}")
print()

# Fit model tốt nhất
elastic_best = ElasticNet(alpha=best_alpha_e, l1_ratio=best_l1, max_iter=10000)
elastic_scores = rmse_cv(elastic_best, X_train, y)
elastic_best.fit(X_train, y)
y_pred_elastic = elastic_best.predict(X_train)

nonzero_e = (elastic_best.coef_ != 0).sum()

print(f"   CV RMSE (5-fold) : {elastic_scores.mean():.5f} ± {elastic_scores.std():.5f}")
print(f"   Train RMSE       : {np.sqrt(mean_squared_error(y, y_pred_elastic)):.5f}")
print(f"   R² (train)       : {r2_score(y, y_pred_elastic):.5f}")
print(f"   Features giữ lại : {nonzero_e} / {X_train.shape[1]}")

---

## 📊 Task 5 — So Sánh Các Models

---

### 🏆 Tổng Kết & Trực Quan Hoá

Sau khi train 4 models, chúng ta sẽ:
1. **Bảng so sánh** RMSE và R²
2. **Bar chart** RMSE CV — model nào tốt nhất?
3. **Scatter plot** Actual vs Predicted — model dự đoán có sát không?
4. **Residuals histogram** — phân phối sai số có chuẩn không?

In [ ]:
# ============================================================
# 📊 TASK 5.1 — BẢNG SO SÁNH & BAR CHART RMSE
# ============================================================

# ── Tổng hợp kết quả ──────────────────────────────────────
models = {
    'Linear Regression': (LinearRegression(),                                          'Không regularization'),
    'Ridge'            : (Ridge(alpha=ridge_cv.alpha_),                                'L2 penalty'),
    'Lasso'            : (Lasso(alpha=lasso_cv.alpha_, max_iter=10000),                'L1 penalty'),
    'ElasticNet'       : (ElasticNet(alpha=best_alpha_e, l1_ratio=best_l1, max_iter=10000), 'L1+L2'),
}

results = {}
print(f"{'Model':<22} {'CV RMSE':>10} {'± Std':>8}  {'Nhận xét'}")
print("─" * 65)

for name, (model, note) in models.items():
    scores = rmse_cv(model, X_train, y)
    results[name] = {'mean': scores.mean(), 'std': scores.std(), 'note': note}
    goal = '✅ ĐẠT' if scores.mean() < 0.12 else '⚠️  chưa đạt'
    print(f"{name:<22} {scores.mean():>10.5f} ±{scores.std():.5f}  {goal}  [{note}]")

print()
best_model_name = min(results, key=lambda k: results[k]['mean'])
print(f"🏆 Model tốt nhất: {best_model_name}  (CV RMSE = {results[best_model_name]['mean']:.5f})")
print(f"🎯 Mục tiêu       : RMSE < 0.12")

# ── Bar Chart RMSE So Sánh ─────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

model_names = list(results.keys())
rmse_values = [results[n]['mean'] for n in model_names]
rmse_stds   = [results[n]['std']  for n in model_names]
colors      = ['#EF5350', '#42A5F5', '#FF7043', '#66BB6A']

bars = ax.bar(model_names, rmse_values, yerr=rmse_stds,
              color=colors, edgecolor='white', linewidth=1.2,
              capsize=5, error_kw={'linewidth': 1.5})

ax.axhline(y=0.12, color='red', linestyle='--', linewidth=1.5,
           label='Mục tiêu RMSE = 0.12', alpha=0.8)

for bar, val in zip(bars, rmse_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylabel('CV RMSE (5-fold)', fontsize=11)
ax.set_title('So Sánh RMSE Các Linear Models\n(Buổi 6 — Cross-Validation 5-fold)', fontsize=12)
ax.legend(fontsize=10)
ax.set_ylim(0, max(rmse_values) * 1.2)
ax.set_xticklabels(model_names, fontsize=10)
plt.tight_layout()
plt.savefig('images/buoi6_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 📊 TASK 5.2 — SCATTER: ACTUAL VS PREDICTED (2x2)
# ============================================================

# Fit tất cả models trên toàn X_train
fitted_models = {}
for name, (model, _) in models.items():
    model.fit(X_train, y)
    fitted_models[name] = model

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle('Actual vs Predicted — So Sánh 4 Linear Models\n(trên tập Train)', fontsize=13)
axes = axes.flatten()

model_colors = ['#EF5350', '#42A5F5', '#FF7043', '#66BB6A']

for idx, (name, model) in enumerate(fitted_models.items()):
    ax     = axes[idx]
    y_pred = model.predict(X_train)

    rmse_val = np.sqrt(mean_squared_error(y, y_pred))
    r2_val   = r2_score(y, y_pred)

    ax.scatter(y, y_pred, alpha=0.3, s=10, color=model_colors[idx])
    lims = [y.min(), y.max()]
    ax.plot(lims, lims, 'k--', linewidth=1.2, label='Perfect prediction')

    ax.set_xlabel('Actual log(SalePrice)', fontsize=9)
    ax.set_ylabel('Predicted log(SalePrice)', fontsize=9)
    ax.set_title(f'{name}\nRMSE={rmse_val:.4f} | R²={r2_val:.4f}', fontsize=10)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('images/buoi6_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

print("💡 Nhận xét: Nếu điểm nằm sát đường đứt (y=x) → model dự đoán chính xác")
print("   Điểm phân tán nhiều → model chưa tốt ở một số vùng giá")

In [ ]:
# ============================================================
# 📊 TASK 5.3 — RESIDUALS ANALYSIS
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Phân Tích Sai Số (Residuals) — Ridge vs Lasso', fontsize=12)

for idx, name in enumerate(['Ridge', 'Lasso']):
    ax        = axes[idx]
    model     = fitted_models[name]
    y_pred    = model.predict(X_train)
    residuals = y - y_pred

    ax.hist(residuals, bins=50,
            color='#42A5F5' if name == 'Ridge' else '#FF7043',
            edgecolor='white', alpha=0.8)
    ax.axvline(x=0, color='black', linestyle='--', linewidth=1.2)
    ax.axvline(x=residuals.mean(), color='red', linestyle='-',
               linewidth=1.2, label=f'Mean = {residuals.mean():.4f}')

    ax.set_xlabel('Residual (Actual − Predicted)', fontsize=10)
    ax.set_ylabel('Số lượng', fontsize=10)
    ax.set_title(f'{name} Residuals\nStd = {residuals.std():.4f}', fontsize=11)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('images/buoi6_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

print("💡 Nhận xét:")
print("   Residuals phân phối CHUẨN (hình chuông, tâm ≈ 0) → model tốt")
print("   Nếu có đuôi dài hoặc lệch → còn pattern chưa được học")

---

### 🔬 Giải Thích — So Sánh Models

---

**Đọc biểu đồ Actual vs Predicted:**

| Điểm nằm ở | Ý nghĩa |
|---|---|
| Trên đường y=x | Model **dự đoán cao hơn** thực tế |
| Dưới đường y=x | Model **dự đoán thấp hơn** thực tế |
| Xa đường y=x | **Sai số lớn** ở những điểm đó |
| Vùng giá trị cao (phải) | Nhà đắt thường dự đoán kém chính xác hơn |

**Đọc biểu đồ Residuals:**

```
Residual tốt:
   ✅ Phân phối chuẩn (hình chuông)
   ✅ Tâm ≈ 0 (không bias)
   ✅ Không có pattern (ngẫu nhiên)

Residual xấu:
   ❌ Lệch trái/phải → bias hệ thống
   ❌ Có hình dạng đặc biệt → model bỏ sót pattern phi tuyến
   ❌ Đuôi dài → outlier chưa xử lý hết
```

**🌐 Các Metric Đánh Giá Khác**

| Metric | Công thức | Ưu điểm | Nhược điểm |
|---|---|---|---|
| **RMSE** | $\sqrt{\frac{\sum(\hat{y}-y)^2}{n}}$ | Cùng đơn vị target | Nhạy với outlier |
| **MAE** | $\frac{\sum|\hat{y}-y|}{n}$ | Dễ giải thích | Không phân biệt sai số lớn/nhỏ |
| **R²** | $1 - \frac{SS_{res}}{SS_{tot}}$ | Chuẩn hoá 0–1 | Không nói được sai số thực tế |
| **RMSLE** | $\sqrt{\frac{\sum(\log\hat{y}-\log y)^2}{n}}$ | Tốt cho data lệch | Khó giải thích trực tiếp |

---

## 🔍 Task 6 — Phân Tích Hệ Số (Coefficient Analysis)

---

### 🤔 Tại Sao Cần Phân Tích Hệ Số?

Một trong những ưu điểm lớn nhất của Linear Models là **khả năng giải thích** (interpretability):

```
Hệ số dương (+): Feature tăng → SalePrice tăng
Hệ số âm  (-): Feature tăng → SalePrice giảm
Hệ số = 0    : Lasso đã loại feature này (không ảnh hưởng)

Ví dụ:
   OverallQual coeff = +0.15 → chất lượng tăng 1 bậc → log(price) tăng 0.15
                              → giá nhà tăng ≈ e^0.15 - 1 ≈ 16%
```

In [ ]:
# ============================================================
# 🔍 TASK 6 — COEFFICIENT ANALYSIS (Ridge vs Lasso)
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(15, 8))
fig.suptitle('Top 20 Features Quan Trọng Nhất\n(Hệ số Ridge và Lasso)', fontsize=13)

for idx, name in enumerate(['Ridge', 'Lasso']):
    ax    = axes[idx]
    model = fitted_models[name]
    coefs = pd.Series(model.coef_, index=X_train.columns)
    top20 = coefs.abs().sort_values(ascending=False).head(20)
    top20_coefs = coefs[top20.index]

    colors = ['#42A5F5' if v > 0 else '#EF5350' for v in top20_coefs.values]
    top20_coefs[::-1].plot(kind='barh', ax=ax, color=colors[::-1], edgecolor='white')

    ax.set_title(f'{name} Regression\n(xanh = dương, đỏ = âm)', fontsize=11)
    ax.set_xlabel('Hệ số (Coefficient)', fontsize=10)
    ax.axvline(x=0, color='black', linewidth=0.8)

    zero_count = (coefs == 0).sum()
    if zero_count > 0:
        ax.text(0.98, 0.02, f'{zero_count} features bị loại (coeff=0)',
                transform=ax.transAxes, ha='right', va='bottom',
                fontsize=8, color='gray',
                bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig('images/buoi6_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

print("💡 Nhận xét:")
print("   Ridge : tất cả features đều có hệ số ≠ 0")
print("   Lasso : một số features bị loại (coeff = 0) — model thưa hơn")
print()
print("   Features có hệ số DƯƠNG lớn → tăng giá nhà mạnh nhất")
print("   Features có hệ số ÂM  lớn → giảm giá nhà mạnh nhất")

---

## 📝 Bài Tập Buổi 6

---

### 🎯 Bài Tập 1 — Ridge với alpha Tùy Chỉnh

**Mục tiêu:** Thử Ridge với `alpha=50` (cao hơn RidgeCV tự chọn) và so sánh RMSE.

**Câu hỏi:**
- RMSE có tốt hơn RidgeCV không? Tại sao?
- alpha lớn hơn → regularization **mạnh** hay **yếu** hơn?

In [ ]:
# ─── Bài Tập 1: Ridge với alpha=50 ────────────────────────

# TODO 1: Tạo Ridge model với alpha=50
ridge_50 = ???(alpha=???)

# TODO 2: Tính CV RMSE sử dụng hàm rmse_cv
scores_50 = rmse_cv(???, X_train, y)

# TODO 3: So sánh với RidgeCV
print(f"Ridge(alpha=50)  CV RMSE : {scores_50.mean():.5f} ± {scores_50.std():.5f}")
print(f"RidgeCV (best α) CV RMSE : {ridge_scores.mean():.5f} ± {ridge_scores.std():.5f}")
print(f"RidgeCV alpha    : {ridge_cv.alpha_:.2f}")
print()

# TODO 4: Giải thích — model nào tốt hơn?
# Trả lời: ...

<details>
<summary>💡 Đáp Án Bài 1</summary>

```python
ridge_50  = Ridge(alpha=50)
scores_50 = rmse_cv(ridge_50, X_train, y)
```

**Giải thích:**
- Alpha lớn → regularization mạnh hơn → hệ số bị nén nhiều hơn
- RidgeCV chọn alpha tối ưu bằng cross-validation → thường tốt hơn alpha tuỳ chỉnh
- Nếu `alpha=50` tệ hơn → chứng tỏ RidgeCV đã tìm đúng mức regularization

</details>

---

### 🎯 Bài Tập 2 — Lasso Feature Selection

**Mục tiêu:** Xem Lasso đã loại features nào và tìm 5 features có tác động âm lớn nhất.

**Câu hỏi:**
- Bao nhiêu features bị loại?
- 5 features nào làm **giảm** giá nhà nhất?

In [ ]:
# ─── Bài Tập 2: Lasso Feature Selection ──────────────────

# TODO 1: Tạo pd.Series chứa hệ số của lasso_best
coef_series = pd.Series(???, index=???)

# TODO 2: Đếm số features bị loại (coeff = 0)
n_removed = (??? == 0).sum()
print(f"Features bị Lasso loại: {n_removed}")

# TODO 3: Top 5 features có hệ số âm nhỏ nhất (ảnh hưởng xấu nhất)
top5_negative = coef_series.sort_values(ascending=???).head(5)
print("\nTop 5 features làm GIẢM giá nhà nhiều nhất:")
print(top5_negative)

<details>
<summary>💡 Đáp Án Bài 2</summary>

```python
coef_series  = pd.Series(lasso_best.coef_, index=X_train.columns)
n_removed    = (coef_series == 0).sum()
top5_negative = coef_series.sort_values(ascending=True).head(5)
```

**Giải thích:**
- `sort_values(ascending=True)` → sắp xếp nhỏ → lớn → hệ số âm nhất đứng đầu
- Features âm lớn = **làm giảm** log(SalePrice) → ảnh hưởng tiêu cực đến giá

</details>

---

### 🎯 Bài Tập 3 — Tính R² và RMSE trên Tập Train

**Mục tiêu:** Dùng `lasso_best` để dự đoán trên `X_train` và tính 2 metrics.

**Câu hỏi:**
- R² của Lasso là bao nhiêu? Đạt mục tiêu >0.90 không?
- RMSE trên train có thấp hơn CV RMSE không? Tại sao?

In [ ]:
# ─── Bài Tập 3: Tính R² và RMSE trên Train ───────────────

# TODO 1: Dự đoán với lasso_best trên tập train
y_pred_bt3 = lasso_best.predict(???)

# TODO 2: Tính R²
r2_bt3 = r2_score(???, ???)

# TODO 3: Tính RMSE (dùng np.sqrt + mean_squared_error)
rmse_bt3 = np.sqrt(mean_squared_error(???, ???))

print(f"Lasso — Train R²   : {r2_bt3:.5f}  {'✅ ĐẠT' if r2_bt3 > 0.90 else '⚠️  chưa đạt'}")
print(f"Lasso — Train RMSE : {rmse_bt3:.5f}  {'✅ ĐẠT' if rmse_bt3 < 0.12 else '⚠️  chưa đạt'}")
print(f"Lasso — CV RMSE    : {lasso_scores.mean():.5f}")
print()
print("CV RMSE thường CAO hơn Train RMSE vì:")
print("  Train RMSE  = đánh giá trên dữ liệu đã học")
print("  CV RMSE     = đánh giá trên dữ liệu chưa thấy → sát thực tế hơn")

<details>
<summary>💡 Đáp Án Bài 3</summary>

```python
y_pred_bt3 = lasso_best.predict(X_train)
r2_bt3     = r2_score(y, y_pred_bt3)
rmse_bt3   = np.sqrt(mean_squared_error(y, y_pred_bt3))
```

**Giải thích:**
- Train RMSE thường thấp hơn CV RMSE vì model đã "thấy" dữ liệu train
- CV RMSE đánh giá trên fold validation → sát với hiệu năng thực tế trên test set
- Nếu Train RMSE << CV RMSE → dấu hiệu **overfitting**

</details>

---

## 🏁 Tổng Kết Buổi 6

---

### ✅ Những Gì Đã Học

| Task | Nội dung | Kết quả |
|------|----------|---------|
| Task 1 | Cross-Validation & Linear Regression | Baseline RMSE |
| Task 2 | Ridge Regression + RidgeCV | L2 regularization |
| Task 3 | Lasso Regression + LassoCV | L1 + feature selection |
| Task 4 | ElasticNet (L1 + L2) | Kết hợp 2 loại |
| Task 5 | So sánh 4 models | Biểu đồ trực quan |
| Task 6 | Coefficient Analysis | Giải thích model |

---

### 📊 Kết Quả Cuối Buổi 6

| Model | Đặc điểm | Khi nào dùng |
|-------|----------|-------------|
| **Linear Regression** | Không regularization, dễ overfit | Baseline |
| **Ridge** | L2 — co đều tất cả hệ số | Khi nhiều features tương quan |
| **Lasso** | L1 — tự chọn features | Khi muốn sparse model |
| **ElasticNet** | L1+L2 — linh hoạt nhất | Khi không chắc dùng loại nào |

---

### 🚀 Buổi 7 — Tree-Based Models

Buổi tiếp theo sẽ khám phá các model mạnh hơn:

```
📌 Decision Tree  — chia dữ liệu theo ngưỡng
📌 Random Forest  — ensemble 100+ cây
📌 Gradient Boosting — học từ lỗi của cây trước
📌 XGBoost / LightGBM — phiên bản tối ưu của GBM
```

> **Mục tiêu Buổi 7:** Đánh bại Linear Models, đạt RMSE < 0.11 và vào Top 25% Kaggle 🏆